In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")
CASE_ROWS = [5192, 40099, 63018]

candidate_paths = list(INPUT.rglob("day9_3_binary_alias_candidate_scores.csv"))
if not candidate_paths:
    raise FileNotFoundError("Attach the saved Day 9.3 notebook output containing the alias candidate score CSV.")
candidate_path = max(candidate_paths, key=lambda p: p.stat().st_mtime)
case_path = candidate_path.parent / "day9_3_binary_alias_case_summary.csv"
if not case_path.is_file():
    raise FileNotFoundError(f"Missing sibling Day 9.3 case summary: {case_path}")
candidate_df = pd.read_csv(candidate_path)
day9_case_df = pd.read_csv(case_path)
if not {"simulation_row", "bic", "fit_period_days", "phase", "duration_days", "noise_sigma"}.issubset(candidate_df.columns):
    raise ValueError("Day 9.3 candidate CSV is missing required columns.")

sources = []
for mp in INPUT.rglob("simulation_metadata.json"):
    if mp.parent.name != "binary":
        continue
    try:
        meta = json.loads(mp.read_text(encoding="utf-8"))
    except Exception:
        continue
    if meta.get("family") == "binary" and int(meta.get("parameter_draws_unique", 0)) >= 100_000:
        sources.append((meta.get("created_utc", ""), mp.parent))
if not sources:
    raise FileNotFoundError("Attach the Day 7 Cosmic swan Bayesian output containing binary simulation files.")
_, BINARY_DIR = sorted(sources, key=lambda x: x[0])[-1]
observations = np.load(BINARY_DIR / "observations.npy", mmap_mode="r")
parameters = pd.read_csv(BINARY_DIR / "parameters.csv")
time_days = np.asarray(np.load(BINARY_DIR / "time_days.npy"), dtype=float)
if observations.shape != (len(parameters), len(time_days)):
    raise ValueError("Binary observations, parameters, and time axis do not align.")
if any(row >= len(parameters) for row in CASE_ROWS):
    raise ValueError("At least one fixed case row is absent from the Day 7 input.")

OUT = WORK / ("day9-4-boundary-refinement-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUT.mkdir(parents=True, exist_ok=False)
print("[PASS] Day 7 binary input:", BINARY_DIR)
print("[PASS] Day 9.3 score input:", candidate_path)
print("[PASS] Cases:", CASE_ROWS, "| output:", OUT)
print("CPU-only; no MCMC sampling.")

In [ ]:
def smooth_clip(x, sharpness=500.0):
    return (np.logaddexp(0.0, sharpness*x)-np.logaddexp(0.0, sharpness*(x-1.0)))/sharpness

def predict(t,p,phase,depth,ratio,duration,ellip,reflection):
    orb=np.mod(t/p-phase+0.5,1.0)-0.5
    sec=np.mod(orb,1.0)-0.5
    half=max(duration/2.0,1e-6)
    ingress=max(half*0.18,1e-6)
    primary=smooth_clip((half+ingress-np.abs(orb))/(2.0*ingress))
    secondary=smooth_clip((half+ingress-np.abs(sec))/(2.0*ingress))
    return 1-depth*primary-depth*ratio*secondary+ellip*np.cos(4*np.pi*orb)+reflection*np.cos(2*np.pi*orb)

case_rows, parameter_rows, plot_data = [], [], []
for row in CASE_ROWS:
    started=time.time()
    matching=candidate_df[candidate_df["simulation_row"].astype(int)==int(row)].sort_values("bic")
    if matching.empty:
        raise ValueError(f"Day 9.3 has no candidate scores for row {row}.")
    prior=matching.iloc[0]
    t=np.asarray(time_days,float)
    y=np.asarray(observations[int(row)],float)
    ok=np.isfinite(t)&np.isfinite(y); t,y=t[ok],y[ok]
    order=np.argsort(t); t,y=t[order],y[order]
    y=y/np.median(y); t=t-t.min()

    center=float(prior["fit_period_days"])
    window=max(0.15,0.02*center)
    p_low=max(0.1,center-window)
    p_high=min(13.0,center+window)
    # Non-period bounds preserve the simulator's declared prior support.
    bounds=[(p_low,p_high),(0.0,1.0),(0.002,0.45),(0.0,1.0),
            (0.005,0.12),(0.0,0.02),(-0.01,0.01),(np.log(1e-4),np.log(0.03))]
    depth0=float(np.clip((np.nanpercentile(y,95)-np.nanpercentile(y,5))/2,0.002,0.45))
    duration0=float(np.clip(float(prior["duration_days"])/center,0.005,0.12))
    noise0=float(np.clip(float(prior["noise_sigma"]),1e-4,0.03))
    phase0=float(np.mod(float(prior["phase"]),1.0))

    def nll(theta):
        pred=predict(t,*theta[:7])
        log_noise=theta[7]
        return float(0.5*np.sum(((y-pred)/np.exp(log_noise))**2+2*log_noise+np.log(2*np.pi)))

    starts=[]
    for shift in (0.0,0.5):
        x=np.array([center,(phase0+shift)%1.0,depth0,0.5,duration0,0.005,0.0,np.log(noise0)])
        starts.append(np.array([np.clip(v,lo+1e-9,hi-1e-9) for v,(lo,hi) in zip(x,bounds)]))
    fits=[minimize(nll,x,method="L-BFGS-B",bounds=bounds,options={"maxiter":1500,"ftol":1e-10}) for x in starts]
    fit=min(fits,key=lambda z:z.fun if np.isfinite(z.fun) else np.inf)
    theta=np.asarray(fit.x,float)
    names=["period_days","phase","depth_primary","depth_ratio","duration_fraction",
           "ellipsoidal_amplitude","reflection_amplitude","log_noise"]
    truth=parameters.iloc[int(row)]
    estimates=dict(zip(names,theta.tolist()))
    estimates["noise_sigma"]=float(np.exp(theta[7]))
    truth_map={name:float(truth[name]) for name in
               ["period_days","phase","depth_primary","depth_ratio","duration_fraction",
                "ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]}
    phase_error=float((estimates["phase"]-truth_map["phase"]+0.5)%1.0-0.5)
    bound_hits=[]
    for name,value,(lo,hi) in zip(names,theta,bounds):
        if min(value-lo,hi-value)/(hi-lo)<0.005:
            bound_hits.append(name)
    prediction=predict(t,*theta[:7])
    noise=float(np.exp(theta[7]))
    row_result={"simulation_row":int(row),"optimizer_success":bool(fit.success),
        "optimizer_message":str(fit.message),"starting_period_days":center,
        "expanded_period_low":p_low,"expanded_period_high":p_high,
        "map_period_days":float(theta[0]),"known_period_days":truth_map["period_days"],
        "period_abs_error_days":abs(float(theta[0])-truth_map["period_days"]),
        "phase_abs_error_cycles":abs(phase_error),
        "nll":float(fit.fun),"map_noise_sigma":noise,
        "rms_residual":float(np.sqrt(np.mean((y-prediction)**2))),
        "rms_over_noise":float(np.sqrt(np.mean((y-prediction)**2))/noise),
        "bound_hits":"|".join(bound_hits),"seconds":round(time.time()-started,1)}
    case_rows.append(row_result)
    for name in ["period_days","phase","depth_primary","depth_ratio","duration_fraction",
                 "ellipsoidal_amplitude","reflection_amplitude","noise_sigma"]:
        estimate=estimates[name]
        target=truth_map[name]
        err=phase_error if name=="phase" else estimate-target
        parameter_rows.append({"simulation_row":int(row),"parameter":name,
                               "map_estimate":estimate,"known_truth":target,
                               "estimate_minus_truth":err,"absolute_error":abs(err)})
    plot_data.append((int(row),t,y,prediction,theta.copy()))
    pd.DataFrame(case_rows).to_csv(OUT/"day9_4_boundary_case_summary.csv",index=False)
    pd.DataFrame(parameter_rows).to_csv(OUT/"day9_4_boundary_parameter_recovery.csv",index=False)
    print(f"row {row}: period {theta[0]:.6f} d; hits={bound_hits or 'none'}; "
          f"RMS/noise={row_result['rms_over_noise']:.3f}; {row_result['seconds']} s",flush=True)

case_df=pd.DataFrame(case_rows)
parameter_df=pd.DataFrame(parameter_rows)

In [ ]:
fig,axes=plt.subplots(len(plot_data),1,figsize=(12,10),squeeze=False)
for ax,(row,t,y,pred,theta) in zip(axes[:,0],plot_data):
    ax.scatter(t,y,s=8,alpha=0.45,label="synthetic observations")
    ax.plot(t,pred,color="crimson",lw=1.0,label="expanded-window MAP fit")
    ax.set_title(f"TIC simulation row {row} | MAP period {theta[0]:.5f} d")
    ax.set_xlabel("Time (days)")
    ax.set_ylabel("Normalized flux")
    ax.legend()
fig.tight_layout()
fig.savefig(OUT/"day9_4_boundary_refinement_fits.png",dpi=150)
plt.show()

summary={"created_utc":datetime.now(timezone.utc).isoformat(),
 "day7_simulation_dir":str(BINARY_DIR),
 "day9_3_input":str(candidate_path),
 "scope":"Three targeted synthetic binary cases refit with an expanded local period window; physical parameter bounds remain at simulator prior support.",
 "case_count":int(len(case_df)),
 "period_hits_search_boundary":int(case_df["bound_hits"].fillna("").str.contains("period").sum()),
 "cases_with_any_parameter_boundary_hit":int(case_df["bound_hits"].fillna("").ne("").sum()),
 "median_period_absolute_error_days":float(case_df["period_abs_error_days"].median()),
 "outputs":["day9_4_boundary_case_summary.csv","day9_4_boundary_parameter_recovery.csv","day9_4_boundary_refinement_fits.png"],
 "interpretation_limits":["Synthetic recovery only; no real-star validation.",
 "The expanded period window is a local follow-up, not a complete global period search.",
 "A MAP fit or optimizer success does not establish posterior convergence or calibrated uncertainty.",
 "Parameters at physical prior limits should be interpreted with the known-truth recovery table, not expanded beyond simulator support."]}
(OUT/"day9_4_boundary_refinement_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(case_df.to_string(index=False))
print(parameter_df.to_string(index=False))
print(json.dumps(summary,indent=2))
print("Outputs:",OUT)